# High Revenue Order Prediction Analysis

## Project Overview
This project focuses on predicting high-revenue orders using product, pricing, and sales channel data. The goal is to identify key factors that contribute to strong sales performance and support data-driven decision-making in pricing and sales strategy.

## Data Description
The dataset contains transactional sales data across multiple sales channels, including order-level details such as product, pricing, quantity, and customer-related identifiers.

### Key Features
- Sales Channel: Method of sale (e.g., Online, In-Store, Wholesale)
- Order Quantity: Number of items in each order
- Unit Cost and Unit Price: Pricing details per product
- Discount Applied: Discount percentage applied to the order
- OrderDate: Date the order was placed

In [69]:
import pandas as pd

df = pd.read_csv('US_Regional_Sales_Data.csv')

print(df.head())
print(df.info())

   OrderNumber Sales Channel WarehouseCode ProcuredDate OrderDate ShipDate  \
0  SO - 000101      In-Store  WARE-UHY1004     31/12/17   31/5/18  14/6/18   
1  SO - 000102        Online  WARE-NMK1003     31/12/17   31/5/18  22/6/18   
2  SO - 000103   Distributor  WARE-UHY1004     31/12/17   31/5/18  21/6/18   
3  SO - 000104     Wholesale  WARE-NMK1003     31/12/17   31/5/18   2/6/18   
4  SO - 000105   Distributor  WARE-NMK1003      10/4/18   31/5/18  16/6/18   

  DeliveryDate CurrencyCode  _SalesTeamID  _CustomerID  _StoreID  _ProductID  \
0      19/6/18          USD             6           15       259          12   
1       2/7/18          USD            14           20       196          27   
2       1/7/18          USD            21           16       213          16   
3       7/6/18          USD            28           48       107          23   
4      26/6/18          USD            22           49       111          26   

   Order Quantity  Discount Applied Unit Cost Unit

## Data Preparation
Data cleaning included converting numeric fields stored as text into proper numerical formats and creating a revenue metric by multiplying order quantity and unit price. A binary target variable was then created to classify orders as high or low revenue based on the median revenue value. Additionally, date features were extracted to support time-based analysis.

In [70]:
df['Unit Cost'] = df['Unit Cost'].str.replace(',', '').astype(float)
df['Unit Price'] = df['Unit Price'].str.replace(',', '').astype(float)

In [71]:
df['Revenue'] = df['Order Quantity'] * df['Unit Price']

In [72]:
df['High_Revenue'] = (df['Revenue'] > df['Revenue'].median()).astype(int)

In [73]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'], dayfirst=True)

df['Order_Month'] = df['OrderDate'].dt.month

/tmp/ipykernel_23864/2170738911.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['OrderDate'] = pd.to_datetime(df['OrderDate'], dayfirst=True)


In [74]:
features = [
    'Order Quantity',
    'Discount Applied',
    'Unit Cost',
    'Order_Month'
]

In [75]:
df = pd.get_dummies(df, columns=['Sales Channel'], drop_first=True)

In [76]:
X = df[features + [col for col in df.columns if 'Sales Channel_' in col]]
y = df['High_Revenue']

## Feature Engineering
Several features were prepared for modeling, including numerical variables such as order quantity, discount applied, unit cost, and order month. Categorical variables such as sales channel were converted into dummy variables to allow inclusion in the model.

## Modeling Approach
A logistic regression model was used to classify orders as high or low revenue. The dataset was split into training and testing sets, and the model was trained using selected numerical and categorical features.

In [77]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## Model Performance
The model achieved an accuracy of approximately 88%, indicating strong classification performance in distinguishing between high and low revenue orders. This level of performance suggests that the selected features provide meaningful information for predicting sales outcomes.

In [78]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8786741713570981


In [79]:
import pandas as pd

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})

print(coefficients.sort_values(by='Coefficient', ascending=False))

                   Feature  Coefficient
0           Order Quantity     1.171372
1         Discount Applied     0.126824
6  Sales Channel_Wholesale     0.123362
4   Sales Channel_In-Store     0.048160
5     Sales Channel_Online     0.012944
2                Unit Cost     0.003254
3              Order_Month    -0.011361


In [80]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[690 106]
 [ 88 715]]


## Key Insights
- Order quantity is the strongest predictor of high-revenue orders, indicating that larger purchase volumes drive overall sales performance
- Sales channel influences revenue outcomes, with wholesale orders showing the highest likelihood of generating high revenue
- Discounting has a negative relationship with revenue performance, suggesting that higher discounts may reduce overall order value
- Unit cost has only a minimal impact on revenue classification, indicating that pricing alone does not determine high-value sales
- Order timing shows limited influence on revenue outcomes, suggesting minimal seasonal impact within the dataset

## Conclusion
The analysis demonstrates that high-revenue orders are primarily driven by order size and sales channel rather than pricing or timing alone. These findings highlight the importance of focusing on volume-driven strategies and channel optimization to maximize revenue performance.

## Business Impact
Understanding the drivers of high-revenue orders can help businesses optimize pricing strategies, prioritize high-performing sales channels, and improve overall revenue generation.